# CUDA Smoke Test (Colab)

This notebook validates that CUDA is available and actually used by PyTorch.
It performs real GPU tensor computation and a tiny training step.

In [1]:
import os
import platform
import subprocess

import torch

print('Python:', platform.python_version())
print('PyTorch:', torch.__version__)
print('CUDA built with PyTorch:', torch.version.cuda)
print('CUDA available:', torch.cuda.is_available())
print('CUDA device count:', torch.cuda.device_count())

if torch.cuda.is_available():
    print('Active GPU:', torch.cuda.get_device_name(0))

print('\n--- nvidia-smi ---')
try:
    out = subprocess.check_output(['nvidia-smi'], stderr=subprocess.STDOUT, text=True)
    print(out[:1200])
except Exception as e:
    print('nvidia-smi not available:', e)

Python: 3.12.13
PyTorch: 2.11.0+cu128
CUDA built with PyTorch: 12.8
CUDA available: True
CUDA device count: 1
Active GPU: Tesla T4

--- nvidia-smi ---
Sat Aug  1 08:40:13 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8             10W /   70W |       3MiB /  15360MiB |      0%     

In [2]:
if not torch.cuda.is_available():
    raise RuntimeError('CUDA is not available. In Colab, switch Runtime -> Change runtime type -> GPU.')

device = torch.device('cuda')
print('Using device:', device)

# Real GPU compute: large matrix multiplication
n = 4096
a = torch.randn(n, n, device=device)
b = torch.randn(n, n, device=device)

torch.cuda.synchronize()
start = torch.cuda.Event(enable_timing=True)
end = torch.cuda.Event(enable_timing=True)

start.record()
c = a @ b
end.record()
torch.cuda.synchronize()

print('Matmul output shape:', tuple(c.shape))
print('Matmul device:', c.device)
print('Matmul elapsed (ms):', round(start.elapsed_time(end), 2))

Using device: cuda
Matmul output shape: (4096, 4096)
Matmul device: cuda:0
Matmul elapsed (ms): 195.3


In [3]:
# Tiny training step on GPU
model = torch.nn.Sequential(
    torch.nn.Linear(256, 128),
    torch.nn.ReLU(),
    torch.nn.Linear(128, 22),
).to('cuda')

x = torch.randn(512, 256, device='cuda')
y = torch.randint(0, 22, (512,), device='cuda')

opt = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = torch.nn.CrossEntropyLoss()

opt.zero_grad()
logits = model(x)
loss = criterion(logits, y)
loss.backward()
opt.step()

print('One GPU training step completed.')
print('Loss:', float(loss.item()))
print('Logits device:', logits.device)

One GPU training step completed.
Loss: 3.129197835922241
Logits device: cuda:0


## Success Criteria

You are good if all these are true:
- CUDA available is True
- Matmul device is cuda:0
- One GPU training step completed without errors